# Whole-Genome Germline Variant Analysis Pipeline

## Germline Variant Calling Pipeline from BAM Files

### Workflow Overview

1. Download and Install Required Tools
2. Upload Input BAM Files
3. Download and Prepare the Reference Genome
4. Add Read Groups
5. Mark PCR Duplicates
6. Base Quality Score Recalibration (BQSR)
7. Germline Variant Discovery with GATK HaplotypeCaller
8. Extract SNPs and INDELs
9. Variant Filtration
10. Functional Annotation with SnpEff
11. Review Final Results

### Download and Install GATK (Genome Analysis Toolkit)

In [1]:
# Download GATK
!wget -q https://github.com/broadinstitute/gatk/releases/download/4.6.2.0/gatk-4.6.2.0.zip

# Unzip
!unzip -q gatk-4.6.2.0.zip

# Add GATK to PATH
import os
os.environ["PATH"] += ":/content/gatk-4.6.2.0"

In [2]:
# 4. Verify installation
!gatk --version | head -n 3

Using GATK jar /content/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /content/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar --version
The Genome Analysis Toolkit (GATK) v4.6.2.0
HTSJDK Version: 4.2.0
Picard Version: 3.4.0


### Install SAMtools for BAM File Processing

In [3]:
!apt-get install -qq -y samtools > /dev/null

## Uploading and unziping Data (BAM file)

In [4]:
%%bash
# Use standard unzip to extract the files cleanly
unzip -qo /content/WES-LUNG.zip -d /content/

## Step 1: Download and Index Reference

In [5]:
%%bash
# 1. Create directory silently
mkdir -p /content/reference_hg19

# 2. Download hg19 fasta completely quiet
wget -q -P /content/reference_hg19/ https://hgdownload.soe.ucsc.edu/goldenPath/hg19/bigZips/hg19.fa.gz

# 3. Decompress quietly
gunzip -q /content/reference_hg19/hg19.fa.gz

# 4. Index the fasta file quietly
samtools faidx /content/reference_hg19/hg19.fa

# 5. Create sequence dictionary and redirect all log outputs to null
./gatk-4.6.2.0/gatk CreateSequenceDictionary -R /content/reference_hg19/hg19.fa > /dev/null 2>&1

In [6]:
import os

ref_dir = "/content/reference_hg19"
expected_files = ["hg19.fa", "hg19.fa.fai", "hg19.dict"]

print("🧬 --- REFERENCE GENOME VERIFICATION --- 🧬\n")

all_exist = True
for file in expected_files:
    path = os.path.join(ref_dir, file)
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"✅ Found: {file:<12} | Size: {size_mb:>8.2f} MB")
    else:
        print(f"❌ Missing: {file}")
        all_exist = False

if all_exist:
    print("\n🚀 hg19 Reference Genome is completely indexed and ready for GATK!")

🧬 --- REFERENCE GENOME VERIFICATION --- 🧬

✅ Found: hg19.fa      | Size:  3051.67 MB
✅ Found: hg19.fa.fai  | Size:     0.00 MB
✅ Found: hg19.dict    | Size:     0.01 MB

🚀 hg19 Reference Genome is completely indexed and ready for GATK!


## Adding Read Groups and Marking Duplicates
- Read Groups (@RG tags) identify the sample name, sequencing platform, and library. We will use GATK's AddOrReplaceReadGroups and sort the file by genomic coordinates.

- During PCR amplification in sequencing, the exact same DNA fragment can be sequenced multiple times. We need to flag these "artifacts" so they don't skew our variant calling statistics.

In [7]:
%%bash
# --- 1. PROCESS THE NORMAL SAMPLE ---
java -jar /content/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar AddOrReplaceReadGroups \
    -I /content/WES-LUNG/NORMAL.bam \
    -O /content/NORMAL.rg.bam \
    -RGID 1 -RGLB WES_Lib -RGPL ILLUMINA -RGPU unit1 -RGSM NORMAL \
    --CREATE_INDEX true > /dev/null 2>&1

java -jar /content/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar MarkDuplicates \
    -I /content/NORMAL.rg.bam \
    -O /content/NORMAL.marked_dups.bam \
    -M /content/normal_metrics.txt \
    --CREATE_INDEX true > /dev/null 2>&1

# --- 2. PROCESS THE TUMOR SAMPLE ---
java -jar /content/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar AddOrReplaceReadGroups \
    -I /content/WES-LUNG/TUMOR.bam \
    -O /content/TUMOR.rg.bam \
    -RGID 2 -RGLB WES_Lib -RGPL ILLUMINA -RGPU unit2 -RGSM TUMOR \
    --CREATE_INDEX true > /dev/null 2>&1

java -jar /content/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar MarkDuplicates \
    -I /content/TUMOR.rg.bam \
    -O /content/TUMOR.marked_dups.bam \
    -M /content/tumor_metrics.txt \
    --CREATE_INDEX true > /dev/null 2>&1

In [8]:
import os

processed_files = [
    "NORMAL.marked_dups.bam", "NORMAL.marked_dups.bai",
    "TUMOR.marked_dups.bam", "TUMOR.marked_dups.bai"
]

print("🧬 --- WES PREPROCESSING VERIFICATION --- 🧬\n")

all_ready = True
for file in processed_files:
    if os.path.exists(file):
        size_gb = os.path.getsize(file) / (1024 ** 3)
        print(f"✅ Generated: {file:<23} | Size: {size_gb:>6.3f} GB")
    else:
        print(f"❌ Missing:   {file}")
        all_ready = False

if all_ready:
    print("\n🚀 Both Normal and Tumor samples are fully preprocessed, indexed, and ready for variant calling!")

🧬 --- WES PREPROCESSING VERIFICATION --- 🧬

✅ Generated: NORMAL.marked_dups.bam  | Size:  0.007 GB
✅ Generated: NORMAL.marked_dups.bai  | Size:  0.001 GB
✅ Generated: TUMOR.marked_dups.bam   | Size:  0.008 GB
✅ Generated: TUMOR.marked_dups.bai   | Size:  0.001 GB

🚀 Both Normal and Tumor samples are fully preprocessed, indexed, and ready for variant calling!


Base Quality Score Recalibration (BQSR)

The sequencing machine often introduces systematic errors when assigning quality scores to bases. BQSR uses a database of known polymorphic sites (like dbSNP) to adjust these quality scores so they reflect the true error probability.

⚠️ As my data is 0.1% subsample, BQSR might struggle or throw warnings due to low data volume, For BQSR, we need to run it like this:

In [9]:
# 1. Download known sites (dbSNP for hg19)
# !wget ftp://ftp.ncbi.nih.gov/snp/organisms/human_9606_b151_GRCh37p13/VCF/All_20180418.vcf.gz

# 2. Build the recalibration table
# !./gatk-4.6.2.0/gatk BaseRecalibrator \
 #    -I /content/WES-LUNG_dedup.bam \
  #   -R hg19.fa \
   #  --known-sites All_20180418.vcf.gz \
    # -O /content/recal_data.table

# 3. Apply the recalibration to the BAM
# !./gatk-4.6.2.0/gatk ApplyBQSR \
  #   -I /content/WES-LUNG_dedup.bam \
   #  -R hg19.fa \
    # --bqsr-recal-file /content/recal_data.table \
    # -O /content/WES-LUNG_final.bam

## Germline Variant Discovery

In [10]:
!java -Xmx4g -jar /content/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar HaplotypeCaller \
    -R /content/reference_hg19/hg19.fa \
    -I /content/NORMAL.marked_dups.bam \
    -O /content/germline_raw.vcf > /dev/null 2>&1

In [11]:
import os

vcf_file = "/content/germline_raw.vcf"

print("🧬 --- GERMLINE VARIANT CALLING VERIFICATION --- 🧬\n")

if os.path.exists(vcf_file):
    size_mb = os.path.getsize(vcf_file) / (1024 * 1024)
    print(f"✅ Generated: {os.path.basename(vcf_file)} | Size: {size_mb:.2f} MB")

    # Safely peak inside the file to count the variants without printing raw text
    with open(vcf_file, 'r') as f:
        variant_count = sum(1 for line in f if not line.startswith('#'))
    print(f"📊 Total Raw Germline Variants Called: {variant_count:,}")
    print("\n🚀 Raw germline mutations successfully stored. Ready for variant filtering!")
else:
    print(f"❌ Error: {vcf_file} was not generated. Check your file paths.")

🧬 --- GERMLINE VARIANT CALLING VERIFICATION --- 🧬

✅ Generated: germline_raw.vcf | Size: 0.05 MB
📊 Total Raw Germline Variants Called: 175

🚀 Raw germline mutations successfully stored. Ready for variant filtering!


## Variant Filtration

In [12]:
%%bash
# 1. Apply Variant Filtration and redirect GATK terminal logging to null
java -jar /content/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar VariantFiltration \
    -R /content/reference_hg19/hg19.fa \
    -V /content/germline_raw.vcf \
    -filter "QUAL < 10.0" --filter-name "LowQual" \
    -O /content/germline_filtered_relaxed.vcf > /dev/null 2>&1

# 2. Isolate header metadata and passing variants cleanly
grep -E '^#|PASS' /content/germline_filtered_relaxed.vcf > /content/germline_final_passed.vcf

In [13]:
import os

raw_vcf = "/content/germline_raw.vcf"
passed_vcf = "/content/germline_final_passed.vcf"

print("🧬 --- VARIANT FILTRATION RECONCILIATION --- 🧬\n")

if os.path.exists(raw_vcf) and os.path.exists(passed_vcf):
    # Count variants (ignoring header rows starting with #)
    with open(raw_vcf, 'r') as f:
        raw_count = sum(1 for line in f if not line.startswith('#'))
    with open(passed_vcf, 'r') as f:
        passed_count = sum(1 for line in f if not line.startswith('#'))

    filtered_out = raw_count - passed_count
    pass_rate = (passed_count / raw_count) * 100 if raw_count > 0 else 0

    # Print a structured report table
    print(f"📊 Total Input Raw Variants:      {raw_count:,}")
    print(f"🧹 Variants Filtered Out (QUAL<10): {filtered_out:,}")
    print(f"🏅 High-Confidence PASS Variants:  {passed_count:,}")
    print(f"📈 Pipeline Retention Rate:        {pass_rate:.1f}%")
    print("\n🚀 Hard filtration complete. Verified clean mutations ready for downstream annotation!")
else:
    print("❌ Error: Could not locate the target VCF files to generate statistics.")

🧬 --- VARIANT FILTRATION RECONCILIATION --- 🧬

📊 Total Input Raw Variants:      175
🧹 Variants Filtered Out (QUAL<10): 0
🏅 High-Confidence PASS Variants:  175
📈 Pipeline Retention Rate:        100.0%

🚀 Hard filtration complete. Verified clean mutations ready for downstream annotation!


## Functional Annotation via SnpEff

In [14]:
%%bash
cd /content/
# 1. Download the latest core bundle completely quiet
wget -q https://downloads.sourceforge.net/project/snpeff/snpEff_latest_core.zip

# 2. Extract files quietly and overwrite existing duplicates without prompting
unzip -qo snpEff_latest_core.zip

# 3. Clean up the zip bundle to keep the directory clean
rm -f snpEff_latest_core.zip

In [15]:
# Functional Annotation of Variants Using SnpEff

!java -Xmx4g -jar /content/snpEff/snpEff.jar \
    hg19 \
    /content/germline_final_passed.vcf \
    > /content/germline_annotated.vcf

## Inspect Variants

In [16]:
import pandas as pd

vcf_path = "/content/germline_annotated.vcf"
germline_variants = []

with open(vcf_path, 'r') as f:
    for line in f:
        if line.startswith('#'):
            continue
        chunks = line.strip().split('\t')
        info = chunks[7]

        if "ANN=" in info:
            ann_field = [x for x in info.split(';') if x.startswith("ANN=")][0]
            first_effect = ann_field.replace("ANN=", "").split(',')[0].split('|')

            gene_name = first_effect[3]
            effect = first_effect[1]
            impact = first_effect[2]

            germline_variants.append([chunks[0], chunks[1], chunks[3], chunks[4], gene_name, effect, impact])

df_germline = pd.DataFrame(germline_variants, columns=['Chrom', 'Position', 'Ref', 'Alt', 'Gene', 'Effect', 'Impact'])

print("=== COMPLETE GERMLINE VARIANT PROFILE (ALL IMPACT LEVELS) ===")
if not df_germline.empty:
    # Display the top 20 variants to see what HaplotypeCaller captured
    print(df_germline.head(20).to_string(index=False))
    print(f"\n📊 Total background variants found in this slice: {len(df_germline)}")
else:
    print("The variant file is completely empty. Double-check if 'germline_final_passed.vcf' contains variants.")

=== COMPLETE GERMLINE VARIANT PROFILE (ALL IMPACT LEVELS) ===
Chrom Position Ref       Alt    Gene              Effect   Impact
chr11 26692481   C         T SLC5A12 3_prime_UTR_variant MODIFIER
chr11 26692594   T TTGTGTGTG SLC5A12 3_prime_UTR_variant MODIFIER
chr11 26692631   A         G SLC5A12 3_prime_UTR_variant MODIFIER
chr11 26692733   T         C SLC5A12  synonymous_variant      LOW
chr11 26692742   G         A SLC5A12  synonymous_variant      LOW
chr11 26692811   C         G SLC5A12      intron_variant MODIFIER
chr11 26692933   A         T SLC5A12      intron_variant MODIFIER
chr11 26692967   C         T SLC5A12      intron_variant MODIFIER
chr11 26694979   A         G SLC5A12  synonymous_variant      LOW
chr11 26702664   C         T SLC5A12  synonymous_variant      LOW
chr11 26705310   A         G SLC5A12  synonymous_variant      LOW
chr11 26707847   G      GTTC SLC5A12      intron_variant MODIFIER
chr11 26724972   T     TACAC SLC5A12      intron_variant MODIFIER
chr11 26724995

## Extract SNP and INDELS

In [17]:
%%bash
# 1. Extract only the Single Nucleotide Polymorphisms (SNPs) cleanly
java -jar /content/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar SelectVariants \
    -V /content/germline_final_passed.vcf \
    -select-type SNP \
    -O /content/germline_snps.vcf > /dev/null 2>&1

# 2. Extract only the Insertions and Deletions (Indels) cleanly
java -jar /content/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar SelectVariants \
    -V /content/germline_final_passed.vcf \
    -select-type INDEL \
    -O /content/germline_indels.vcf > /dev/null 2>&1

In [18]:
# Annotate SNPs
!java -Xmx4g -jar /content/snpEff/snpEff.jar hg19 /content/germline_snps.vcf > /content/germline_snps_annotated.vcf

# Annotate Indels
!java -Xmx4g -jar /content/snpEff/snpEff.jar hg19 /content/germline_indels.vcf > /content/germline_indels_annotated.vcf

In [19]:
import pandas as pd

def parse_vcf_to_df(vcf_path):
    variants = []
    with open(vcf_path, 'r') as f:
        for line in f:
            if line.startswith('#'): continue
            chunks = line.strip().split('\t')
            info = chunks[7]
            if "ANN=" in info:
                ann_field = [x for x in info.split(';') if x.startswith("ANN=")][0]
                first_effect = ann_field.replace("ANN=", "").split(',')[0].split('|')
                gene_name = first_effect[3]
                effect = first_effect[1]
                impact = first_effect[2]
                variants.append([chunks[0], chunks[1], chunks[3], chunks[4], gene_name, effect, impact])
    return pd.DataFrame(variants, columns=['Chrom', 'Position', 'Ref', 'Alt', 'Gene', 'Effect', 'Impact'])

# Generate individual dataframes
df_snps = parse_vcf_to_df("/content/germline_snps_annotated.vcf")
df_indels = parse_vcf_to_df("/content/germline_indels_annotated.vcf")

# --- DISPLAY RESULTS ---
print(f"=== GERMLINE SNPs DISCOVERED ({len(df_snps)} total) ===")
print(df_snps.head(10).to_string(index=False))

print("\n" + "="*60 + "\n")

print(f"=== GERMLINE INDELS DISCOVERED ({len(df_indels)} total) ===")
print(df_indels.head(10).to_string(index=False))

# Optional: Export them as clean spreadsheets!
df_snps.to_csv("/content/germline_snps_summary.csv", index=False)
df_indels.to_csv("/content/germline_indels_summary.csv", index=False)
print("\n💾 Saved summaries to 'germline_snps_summary.csv' and 'germline_indels_summary.csv'!")

=== GERMLINE SNPs DISCOVERED (158 total) ===
Chrom Position Ref Alt    Gene              Effect   Impact
chr11 26692481   C   T SLC5A12 3_prime_UTR_variant MODIFIER
chr11 26692631   A   G SLC5A12 3_prime_UTR_variant MODIFIER
chr11 26692733   T   C SLC5A12  synonymous_variant      LOW
chr11 26692742   G   A SLC5A12  synonymous_variant      LOW
chr11 26692811   C   G SLC5A12      intron_variant MODIFIER
chr11 26692933   A   T SLC5A12      intron_variant MODIFIER
chr11 26692967   C   T SLC5A12      intron_variant MODIFIER
chr11 26694979   A   G SLC5A12  synonymous_variant      LOW
chr11 26702664   C   T SLC5A12  synonymous_variant      LOW
chr11 26705310   A   G SLC5A12  synonymous_variant      LOW


=== GERMLINE INDELS DISCOVERED (17 total) ===
Chrom  Position   Ref       Alt    Gene                         Effect   Impact
chr11  26692594     T TTGTGTGTG SLC5A12            3_prime_UTR_variant MODIFIER
chr11  26707847     G      GTTC SLC5A12                 intron_variant MODIFIER
chr11  

## Conclusion and Key Findings

In this study, germline variant calling was performed using a standard GATK-based pipeline followed by functional annotation with SnpEff.

A total of **158 SNPs** and **17 INDELs** were identified from the input BAM-derived VCF files. The majority of variants were located in **non-coding regions**, including intronic and 3′ untranslated regions (3′ UTRs), and were classified as **MODIFIER or LOW impact**, suggesting limited predicted functional consequence.

Notably, most variants were mapped to genes such as *SLC5A12*, with a high proportion of synonymous and intronic substitutions, indicating strong evolutionary conservation of coding regions in this locus. A small number of INDELs showed **MODERATE impact**, including an in-frame insertion in *FAM83G*, which may warrant further functional investigation.

Overall, this dataset reflects a typical germline variant profile dominated by non-coding and low-impact changes. These results provide a foundation for downstream analyses such as population comparison, disease association studies, or integration with transcriptomic data.

The processed results were successfully exported as:
- `germline_snps_summary.csv`
- `germline_indels_summary.csv`